# 09. Praca domowa: rozwiązania

Czas: ok. 30 min na porównanie z własnym kodem

Te same zadania co w notebooku 08 (Praca domowa), tym razem z pełnymi rozwiązaniami i komentarzami. Najpierw spróbuj sam w notebooku 08 (Praca domowa), a tutaj porównaj.
Twój kod może wyglądać inaczej i być poprawny: liczy się wynik i to, że rozumiesz każdą linię.

**Czego się nauczysz**
- użyć własnej funkcji (`def`) na całej kolumnie tabeli (`apply`) i policzyć wynik,
- skleić kilka podsumowań w jedną tabelę do raportu (`pd.concat`),
- zliczyć pary wartości (`pd.crosstab`) i zrobić tabelę przestawną (`pivot_table`),
- rozbić kilka kodów odpadów z jednej komórki na osobne wiersze (`explode`),
- sprawdzić własny plik Excel, zanim puścisz na nim notebooki z kursu.

## 0. Wczytanie czystego pliku

Pracujemy na pliku CZYSTYM `data/przetargi_clean.xlsx`, czyli wyniku notebooka 06 (Pandas: czyszczenie danych).
Cenę za Mg liczymy tak samo jak w notebooku 07 (Pandas: analiza): wartość umowy / wolumen, bez przeliczania na rok; cena powyżej 3000 zł to błąd danych i zamieniamy ją na puste pole.

In [ ]:
import pandas as pd   # pandas = biblioteka do tabel; skrót "pd" jak w całym kursie

# wczytujemy czysty plik do tabeli df (df = przyjęta nazwa tabeli w pandas; ścieżka względem folderu kursu, w VS Code otwórz CAŁY folder)
df = pd.read_excel("data/przetargi_clean.xlsx")

# Excel nie pamięta typu Int64, więc liczba ofert wraca jako 1.0, 2.0; astype("Int64") = z powrotem liczba całkowita, która toleruje puste pola
df["liczba_ofert"] = df["liczba_ofert"].astype("Int64")

# cena za Mg = wartość umowy / wolumen; powyżej 3000 zł to błąd danych, więc zamieniamy na puste pole
df["cena_za_mg"] = df["wartosc_pln"] / df["wolumen_mg"]
# df.loc[warunek, "kolumna"] = wartość: wpisuje wartość TYLKO w wierszach, gdzie warunek ma True; None = puste pole
df.loc[df["cena_za_mg"] > 3000, "cena_za_mg"] = None

print("Wiersze:", len(df), "| kolumny:", len(df.columns))
df.head(3)

In [ ]:
# do analiz liczby ofert bierzemy tylko wiersze ze ZNANĄ liczbą ofert; to jedno z założeń przyjętych w notebooku 07 (Pandas: analiza)
# Pułapka: puste pole <NA> wstawione do if daje błąd, dlatego braki usuwamy ZANIM użyjemy własnej funkcji
offers = df.dropna(subset=["liczba_ofert"]).copy()   # dropna(subset=[kolumna]) = wyrzuć wiersze z pustym polem w tej kolumnie; copy() = osobna kopia, bo zaraz dopiszemy nowe kolumny
print("Przetargi ze znaną liczbą ofert:", len(offers), "z", len(df))

## Zadanie 1. Powtórka z Pythona: poziom konkurencji

Funkcja `competition_level` z notebooka 04 (Funkcje), sekcja 3 (Funkcja z if: poziom konkurencji), zamienia liczbę ofert na opis słowny. Użyj jej na całej kolumnie `liczba_ofert` w tabeli `offers`: `apply` wywołuje funkcję dla każdej komórki kolumny i oddaje nową kolumnę wyników. Potem policz, ile przetargów ma każdy poziom.
- Wpisz funkcję jeszcze raz: 1 to "brak konkurencji", 2 to "słaba konkurencja", 3 i więcej to "konkurencja".
- `apply` daje nową kolumnę `poziom_konkurencji`, a `value_counts()` zlicza opisy.

In [ ]:
# Rozwiązanie 1a: def = przepis dla JEDNEJ wartości (liczby ofert); if / elif / else wybiera opis
def competition_level(offers_count):
    if offers_count == 1:              # przypomnienie: == porównuje, = przypisuje
        return "brak konkurencji"
    elif offers_count == 2:
        return "słaba konkurencja"
    else:                              # 3 i więcej ofert
        return "konkurencja"

# szybki test na pojedynczych liczbach, zanim puścimy funkcję na całej kolumnie
print(competition_level(1), "|", competition_level(2), "|", competition_level(4))

In [ ]:
# Rozwiązanie 1b: apply = wywołaj funkcję dla każdej komórki kolumny; wynik to nowa kolumna
offers["poziom_konkurencji"] = offers["liczba_ofert"].apply(competition_level)
# value_counts liczy, ile razy wystąpił każdy opis
offers["poziom_konkurencji"].value_counts()

In [ ]:
# to samo jako udziały: normalize=True daje ułamki, razy 100 i round(1) = czytelne procenty
(offers["poziom_konkurencji"].value_counts(normalize=True) * 100).round(1)

## Zadanie 2. Województwa: trzy liczby w jednej tabeli

Dla każdego województwa policz: liczbę przetargów, medianę ceny za Mg i udział przetargów z jedną ofertą. Zrób trzy osobne `groupby`, a potem sklej wyniki w jedną tabelę: `pd.concat([...], axis=1)`.
- `axis=1` znaczy "sklej OBOK siebie, jako kolumny"; pandas sam dopasuje wiersze po nazwie województwa (jak WYSZUKAJ.PIONOWO w Excelu).
- Udział jednej oferty: porównanie `== 1` na całej kolumnie daje kolumnę `jedna_oferta` z True/False, a `mean()` z niej to udział True, bo True liczy się jak 1, a False jak 0.

In [ ]:
# Rozwiązanie 2a (1/3): liczba przetargów per województwo; size() liczy wiersze w każdej grupie (także z brakami)
tenders_per_region = df.groupby("wojewodztwo").size()
tenders_per_region.head()

In [ ]:
# Rozwiązanie 2a (2/3): mediana ceny za Mg per województwo; puste ceny median() pomija sam
median_price_per_region = df.groupby("wojewodztwo")["cena_za_mg"].median()
median_price_per_region.round(0).head()

In [ ]:
# Rozwiązanie 2a (3/3): porównanie == 1 daje kolumnę True/False; mean() z niej = udział True (True liczy się jak 1, False jak 0)
offers["jedna_oferta"] = offers["liczba_ofert"] == 1
single_bid_share_per_region = offers.groupby("wojewodztwo")["jedna_oferta"].mean()
single_bid_share_per_region.round(2).head()

In [ ]:
# Rozwiązanie 2b: concat z axis=1 skleja trzy wyniki OBOK siebie; wiersze dopasowuje po nazwie województwa
region_table = pd.concat([tenders_per_region, median_price_per_region, single_bid_share_per_region], axis=1)
# nadajemy kolumnom czytelne nazwy: lista nazw w tej samej kolejności co wyżej
region_table.columns = ["liczba_przetargow", "mediana_ceny_za_mg", "udzial_1_oferty"]
# udział w procentach i pełne złote czyta się lepiej w raporcie
region_table["udzial_1_oferty"] = (region_table["udzial_1_oferty"] * 100).round(1)
region_table["mediana_ceny_za_mg"] = region_table["mediana_ceny_za_mg"].round(0)
# sortujemy malejąco po udziale jednej oferty: gdzie konkurencji brakuje najczęściej
region_table.sort_values("udzial_1_oferty", ascending=False)

## Zadanie 3. Tryb a liczba ofert

`pd.crosstab` zlicza pary wartości: ile przetargów ma dany tryb I daną liczbę ofert (tabela przestawna z samym zliczaniem). W nawiasie podajesz dwie kolumny: pierwsza idzie do wierszy (`tryb`), druga do kolumn (`liczba_ofert`); potem zrób wersję z `normalize="index"` (udziały w wierszu).
- W naszym zbiorze są dwa tryby: `przetarg nieograniczony` i `tryb podstawowy` (rzadszy, w danych dopiero od 2021), więc tabela ma dwa wiersze. Pytanie: czy w trybie podstawowym jedna oferta zdarza się częściej?
- Dla wprawy podstaw potem `wojewodztwo` zamiast `tryb`: kod zostaje ten sam, wierszy będzie 16.

In [ ]:
# Rozwiązanie 3: pierwsza kolumna w nawiasie = wiersze (tryb), druga = kolumny (liczba ofert), w środku liczba przetargów
pd.crosstab(offers["tryb"], offers["liczba_ofert"])

In [ ]:
# normalize="index" zamienia liczby na udziały w wierszu (każdy wiersz sumuje się do 1); razy 100 = procenty
(pd.crosstab(offers["tryb"], offers["liczba_ofert"], normalize="index") * 100).round(1)

W obu trybach jedna oferta to około połowa przetargów (54,6% i 51,6%), a 4 i 5 ofert zdarza się tylko w przetargu nieograniczonym.
Tryb podstawowy ma tu tylko 62 przetargi (wobec 366), więc różnica kilku punktów procentowych to za mało na wniosek; do raportu wystarczy zdanie, że tryb nie zmienia obrazu konkurencji.

In [ ]:
# ten sam kod z województwem w wierszach: 16 wierszy zamiast 2; szukaj województw z największą liczbą w kolumnie 1 (jedna oferta)
pd.crosstab(offers["wojewodztwo"], offers["liczba_ofert"])

## Zadanie 4. Okres umowy a liczba ofert

Czy dłuższe umowy przyciągają więcej ofert? Pogrupuj `offers` po `okres_mies` i policz średnią `liczba_ofert` w każdej grupie (`groupby` = pogrupuj, `mean()` = średnia w każdej grupie).
- Sprawdź też, ile przetargów jest w każdej grupie (`size()`): średnia z kilku przetargów mało znaczy.
- Porównaj wynik z korelacją okresu i liczby ofert z notebooka 07 (Pandas: analiza), sekcja 5 (Korelacje): wyszło tam ok. +0,05, czyli brak związku.

In [ ]:
# Rozwiązanie 4: grupujemy po okresie umowy, liczymy średnią liczby ofert w każdej grupie; round(2) dla czytelności
mean_offers_per_period = offers.groupby("okres_mies")["liczba_ofert"].mean().round(2)
mean_offers_per_period

In [ ]:
# ile przetargów stoi za każdą średnią: size() i sklejenie przez concat tak jak w zadaniu 2 (Województwa: trzy liczby w jednej tabeli)
tenders_per_period = offers.groupby("okres_mies").size()
period_table = pd.concat([mean_offers_per_period, tenders_per_period], axis=1)
period_table.columns = ["srednia_ofert", "liczba_przetargow"]
period_table

Różnice są małe (od ok. 1,4 do 1,7 oferty) i nie układają się w jeden kierunek, a grupa 48 miesięcy to tylko 25 przetargów.
To zgadza się z korelacją z notebooka 07 (Pandas: analiza), sekcja 5 (Korelacje): okres a liczba ofert to ok. +0,05, a korelacja poniżej 0,1 (licząc bez znaku) to brak związku.

## Zadanie 5. Kody odpadów: najczęstsze kody

W `kody_odpadow` jest kilka kodów w jednej komórce, rozdzielonych średnikiem i spacją. Rozbij je: `str.split(separator)` tnie tekst po separatorze i robi z niego listę (kilka wartości w jednej komórce, w nawiasach kwadratowych), `explode()` rozkłada listę na osobne wiersze, `value_counts()` liczy kody.
- Uwaga: cena za Mg dotyczy CAŁEGO przetargu, nie jednego kodu. Po `explode` nie licz z niej średnich, bo ten sam przetarg policzyłby się kilka razy.

In [ ]:
# Rozwiązanie 5 (1/3): str.split tnie tekst po separatorze; w każdej komórce powstaje lista kodów (w nawiasach kwadratowych)
# w wyniku teksty są w pojedynczych cudzysłowach '20 03 01'; to to samo, co "20 03 01" w kodzie
codes_lists = df["kody_odpadow"].str.split("; ")
codes_lists.head(3)

In [ ]:
# Rozwiązanie 5 (2/3): explode "rozkłada" listy: każdy kod trafia do osobnego wiersza
# numer wiersza po lewej się powtarza, bo kilka kodów pochodzi z tego samego przetargu
codes = codes_lists.explode()
print("Kodów łącznie:", len(codes), "| przetargów:", len(df))
codes.head(8)

In [ ]:
# Rozwiązanie 5 (3/3): value_counts liczy, ile razy wystąpił każdy kod; 20 03 01 (odpady zmieszane) jest w każdym przetargu
top_codes = codes.value_counts().head(10)
top_codes

In [ ]:
# udział przetargów z danym kodem: wystąpienia / liczba przetargów, w procentach (kod pojawia się w przetargu najwyżej raz)
# NIE liczymy tu ceny per kod: cena dotyczy całego przetargu, po explode ten sam przetarg wystąpiłby kilka razy
(top_codes / len(df) * 100).round(1)

## Zadanie 6. Top 5 wykonawców w 2024 wg wartości umów

Wybierz przetargi ogłoszone w 2024 (filtr po `rok`), zsumuj `wartosc_pln` per wykonawca, posortuj malejąco i pokaż 5 pierwszych.
- Podpowiedź: `groupby("wykonawca")["wartosc_pln"].sum()`, potem `sort_values(ascending=False)` i `head(5)`.
- Kwoty pokaż w mln zł (podziel przez `1_000_000` i `round(1)`); bez tego pandas pokaże 591 728 796 zł jako `5.917288e+08` (zapis naukowy: przesuń przecinek o 8 miejsc w prawo).

In [ ]:
# Rozwiązanie 6 (1/3): filtr = warunek w nawiasie (True/False dla każdego wiersza), zostają wiersze z True; rok to liczba, więc 2024 bez cudzysłowu
tenders_2024 = df[df["rok"] == 2024]
print("Przetargów w 2024:", len(tenders_2024))

In [ ]:
# Rozwiązanie 6 (2/3): suma wartości umów per wykonawca (wiersze bez wykonawcy groupby pomija)
value_by_contractor = tenders_2024.groupby("wykonawca")["wartosc_pln"].sum()
# ascending=False = od największej; head(5) = pięć pierwszych
top_contractors_2024 = value_by_contractor.sort_values(ascending=False).head(5)
# dzielimy przez milion i zaokrąglamy: kwoty w mln zł (patrz opis zadania)
(top_contractors_2024 / 1_000_000).round(1)

In [ ]:
# Rozwiązanie 6 (3/3): udział w łącznej wartości umów z 2024 ("rynek" = wszystkie przetargi z tego roku; każda MZK <Gmina> to osobna firma)
total_value_2024 = tenders_2024["wartosc_pln"].sum()
(top_contractors_2024 / total_value_2024 * 100).round(1)

In [ ]:
# dla porównania: ranking wg LICZBY wygranych przetargów; kto wygrywa dużo małych, a kto mało dużych
tenders_2024["wykonawca"].value_counts().head(5)

## Zadanie 7 (bonus). Tabela przestawna: mediana ceny per rok i województwo

`pivot_table` to tabela przestawna z Excela: `index` = wiersze, `columns` = kolumny, `values` = co liczymy, `aggfunc` = jak liczymy.
Zrób: `df.pivot_table(values="cena_za_mg", index="rok", columns="wojewodztwo", aggfunc="median")`. Puste pole (`NaN`) znaczy: w tym roku nie było w tym województwie przetargu ze znaną ceną.

In [ ]:
# Rozwiązanie 7: cztery ustawienia w nawiasie: co liczymy (values), co w wierszach (index), co w kolumnach (columns), jak liczymy (aggfunc)
price_pivot = df.pivot_table(values="cena_za_mg", index="rok", columns="wojewodztwo", aggfunc="median")
# round(0) = pełne złote; NaN = brak przetargu ze znaną ceną w tej parze rok/województwo
price_pivot.round(0)

In [ ]:
# ta sama tabela z zamienionymi miejscami index i columns: 16 wierszy i 6 kolumn lepiej mieści się w raporcie
df.pivot_table(values="cena_za_mg", index="wojewodztwo", columns="rok", aggfunc="median").round(0)

## Zadanie 8. Checklista: podmieniam na swoje dane

Otwórz notebook 06 (Pandas: czyszczenie danych) obok własnego pliku Excel i wypisz w komentarzach, co musisz zmienić (lista jest w jego sekcji 12 (Jak podmienić na swoje dane)): ścieżka, nazwy kolumn (słownik `COLUMN_NAMES`), format daty, jednostki (Mg czy m³, czyli tony czy metry sześcienne; brutto czy netto; za umowę czy za rok).
Potem wczytaj plik i uruchom trzy kontrole: `info()`, `isna().sum()`, `describe()`. Na trening użyj `data/przetargi_clean.xlsx`.

**Checklista "podmieniam na swoje dane"** (do zrobienia w notebooku 06 (Pandas: czyszczenie danych), w tej kolejności):
1. Ścieżka: `pd.read_excel("data/moj_plik.xlsx")`; plik wrzuć do folderu `data`; jeśli arkuszy jest kilka, dodaj `sheet_name="nazwa"`.
2. Nazwy kolumn: w słowniku `COLUMN_NAMES` lewa strona (klucz) DOKŁADNIE jak nagłówek w Twoim Excelu (spacje, wielkość liter), prawa strona (wartość) bez zmian.
3. Format daty: `format="%d.%m.%Y"` pasuje do `15.03.2021`; dla `2021-03-15` wpisz `"%Y-%m-%d"`.
4. Jednostki: wolumen w Mg (tonach) czy m³ (metrach sześciennych); wartość brutto czy netto; za całą umowę czy za rok. Inaczej cena za Mg nie ma sensu.
5. `info()`: kolumna z liczbami ma typ `object` albo `str` (jedno i drugie = tekst)? To kolumna do czyszczenia funkcją `clean_amount`.
6. `isna().sum()`: kolumna prawie cała pusta? W Excelu sprawdź, czy nagłówek nie jest w drugim wierszu (wtedy `pd.read_excel(..., header=1)`; pandas liczy wiersze od 0, więc 1 to drugi wiersz) albo czy komórki nie są scalone.
7. Zła nazwa w słowniku `COLUMN_NAMES` objawia się inaczej: błędem `KeyError` (nie ma takiej kolumny) przy pierwszym użyciu nowej nazwy w dalszym kodzie.
8. `describe()`: patrz na `min` i `max`: zera, wartości ujemne albo 10 razy za duże to błąd danych, tak jak nasze ceny powyżej 3000 zł za Mg (wartości odstające, czyli skrajne liczby wzięte z błędu w danych).

In [ ]:
# Rozwiązanie 8, kontrola 1: TRENING na czystym pliku; u siebie podmień ścieżkę (i ewentualnie sheet_name)
my_df = pd.read_excel("data/przetargi_clean.xlsx")
# info(): nazwy kolumn, ile pól wypełnionych, typ; object (albo str) = tekst; tekst tam, gdzie ma być liczba = do czyszczenia
my_df.info()

In [ ]:
# kontrola 2: ile braków w każdej kolumnie; kolumna prawie cała pusta = sprawdź w Excelu nagłówek (drugi wiersz?) i scalone komórki
my_df.isna().sum()

In [ ]:
# kontrola 3: statystyki kolumn liczbowych; include="number" = tylko kolumny liczbowe (bez tego pandas dodałby też kolumnę z datą)
# patrz na min i max: zero, ujemne albo 10 razy za duże to sygnał błędu
# round(0) = pełne złote zamiast zapisu naukowego typu 1.6e+06 (czyli 1 600 000)
my_df.describe(include="number").round(0)

**Podsumowanie**
- Wzorzec do raportu: kilka osobnych `groupby`, potem `pd.concat([...], axis=1)`; działa tak samo po roku, województwie czy okresie umowy.
- `str.split` + `explode` + `value_counts` to sposób na "kilka wartości w jednej komórce"; statystyki per przetarg liczymy PRZED `explode`.
- `pd.crosstab` zlicza pary wartości, `pivot_table` to tabela przestawna z czterema ustawieniami w nawiasie.
- Checklista z zadania 8 to Twój start z własnymi danymi; potem notebooki 06 (Pandas: czyszczenie danych) i 07 (Pandas: analiza) działają bez zmian w kodzie.

To koniec materiałów z pierwszych zajęć. Co dalej, ustalamy po pracy domowej.